In [1]:
# Assign the RR to PWM PM2.5

In [1]:
import os
import xarray as xr
import numpy as np

In [2]:
# === Path config ===
PM_DIR = "/glade/work/awells/air_quality/CESM/pm25/exposure/"
RR_DIR = "/glade/work/awells/air_quality/GBD19_burdendata/"

In [17]:
# === Health variables ===
# COPD (chronic obstructive pulonary disease)
# LRI (lower respiratory infection)
# IHD (ischemic heart disease)
# DM2 (type 2 diabetes)
# LC (tracheal, bronchus, and lung cancer)
# Stroke
health_vars = ["DM", "LC", "COPD", "LRI"]
age_health_vars = ["Stroke", "IHD"]

In [20]:
# === Main loop ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/pm25/RR/"
SCENARIOS = ["ARISE", "SSP245"]

for health_VAR in health_vars:
    for scenario in SCENARIOS:
        for ens_num in range(1, 11):
            print(f"Processing {scenario} ensemble member {ens_num:02d} for {health_VAR}")
            # Load data arrays
            if scenario == "ARISE":
                dates = "2035-2069"
            elif scenario == "SSP245":
                dates = "2020-2069"

            pm25_file = f"PM25_country_population_weighted_exposure_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            pm25_path = os.path.join(PM_DIR, pm25_file)
            pm25 = xr.open_dataarray(pm25_path)

            RR_file = f"MRBRT_{health_VAR}.nc"
            RR_path = os.path.join(RR_DIR, RR_file)
            RR_list = xr.open_dataarray(RR_path)

            RR = RR_list.sel(pm25=pm25, method="nearest")

            # Extract components
            RR_mean = RR.sel(column=health_VAR)  # mean
            RR_lower = RR.sel(column="95% UI")  # lower bound
            RR_upper = RR.sel(column="Unnamed: 2")  # upper bound

            # Estimate standard deviation from the 95% confidence interval
            # z-score for 97.5% in normal dist ≈ 1.96
            RR_std = (RR_upper - RR_lower) / (2 * 1.96)

            # Now draw 1000 samples from a normal distribution for each [country, year]
            # Output shape: (country:204, year:35, sample:1000)

            samples = np.random.normal(
                loc=RR_mean.values[..., np.newaxis],       # mean
                scale=RR_std.values[..., np.newaxis],      # std
                size=(len(pm25.country), len(pm25.year), 1000)                        # output shape
            )

            # Optionally: prevent negative values if RR must be > 0
            samples = np.clip(samples, a_min=0, a_max=None)

            # Convert to xarray for ease of use
            RR_samples = xr.DataArray(
                samples,
                dims=("country", "year", "sample"),
                coords={
                    "country": RR.country,
                    "year": RR.year,
                    "sample": np.arange(1000)
                }
            )

            description = (f"Country level Relative Risk value for {scenario} "
                           f"ensemble member {ens_num:02d} based on McDuffie "
                           "et al. (2021) - scripts by A.F. Wells (2025)")

            RR_samples.attrs["scenario"] = scenario
            RR_samples.attrs["ensemble"] = ens_num
            RR_samples.attrs["description"] = description

            out_file = f"RR_{health_VAR}_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            RR_samples.to_netcdf(out_path)

Processing ARISE ensemble member 01 for DM
Processing ARISE ensemble member 02 for DM
Processing ARISE ensemble member 03 for DM
Processing ARISE ensemble member 04 for DM
Processing ARISE ensemble member 05 for DM
Processing ARISE ensemble member 06 for DM
Processing ARISE ensemble member 07 for DM
Processing ARISE ensemble member 08 for DM
Processing ARISE ensemble member 09 for DM
Processing ARISE ensemble member 10 for DM
Processing SSP245 ensemble member 01 for DM
Processing SSP245 ensemble member 02 for DM
Processing SSP245 ensemble member 03 for DM
Processing SSP245 ensemble member 04 for DM
Processing SSP245 ensemble member 05 for DM
Processing SSP245 ensemble member 06 for DM
Processing SSP245 ensemble member 07 for DM
Processing SSP245 ensemble member 08 for DM
Processing SSP245 ensemble member 09 for DM
Processing SSP245 ensemble member 10 for DM
Processing ARISE ensemble member 01 for LC
Processing ARISE ensemble member 02 for LC
Processing ARISE ensemble member 03 for LC
P

KeyboardInterrupt: 